In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname



root_path = dirname(os.getcwd()) + "/AdaTest"

pd.set_option("display.max_columns", None)
# data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/comuzzi/_processed/"
data_dir_graphs = root_path + "/data/datasets/comuzzi/graphs_repair/"

print(root_path, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = "cpu"

/home/sebastiano.dissegna/AdaTest
/home/sebastiano.dissegna/AdaTest/data/datasets/comuzzi/_processed/
/home/sebastiano.dissegna/AdaTest/data/datasets/comuzzi/graphs_repair/


In [2]:
#ACT_TIME_ONLY = True

In [3]:
ACT_TIME_ONLY = False

In [4]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [5]:
list(datasets_info.keys())

['BPI_Challenge_2013_open_problems',
 'sp2020',
 'Helpdesk',
 'BPI20_RequestForPayment',
 'BPI Challenge 2017 - Offer log',
 'BPI_Challenge_2012_W_Complete',
 'BPI_Challenge_2012_A',
 'bpi_2012_CZ',
 'bpi_2013_CZ',
 'large_log_CZ',
 'small_log_CZ',
 'sp2020_CZ',
 'BPI20_RequestForPayment_CZ']

In [6]:
dataset = "bpi_2013_CZ"

In [7]:
with open("data/dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[dataset]

In [8]:
tab_all = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_all.csv") 
tab_all.head()

,CaseID,Activity,org:resource,time:timestamp,concept:name,impact,lifecycle:transition,org:group,org:role,organization country,organization involved,product,resource country
0,1,Queued-Awaiting Assignment,Value 1,0.000000,Queued,High,Awaiting Assignment,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA
1,1,Accepted-In Progress,Value 1,19.087576,Accepted,High,In Progress,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA
2,1,Accepted-Assigned,Value 1,19.087576,Accepted,High,Assigned,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA
3,1,Accepted-In Progress,Value 1,19.087580,Accepted,High,In Progress,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA
4,1,Completed-Closed,Value 1,19.087581,Completed,High,Closed,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA


In [9]:
nan_methods = ["odd", "even", "random", "window","attr_level"]

masked_datasets = {key : pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_masked_{key}_all.csv") for key in nan_methods}

In [10]:
tab_train = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_test.csv")

In [11]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)
for k in masked_datasets:
    masked_datasets[k]["CaseID"] = masked_datasets[k]["CaseID"].astype(np.str_)

In [12]:
from data.utils import get_case_ids

CASE_TRAIN_IDS = get_case_ids(tab_train)
CASE_VALID_IDS = get_case_ids(tab_valid)
CASE_TEST_IDS = get_case_ids(tab_test)

In [13]:
def get_new_timestamp(trace: pd.DataFrame):
    times = list(trace["time:timestamp"].copy())
    for i in range(1,len(times)):
        times[i] = times[i] - times[0]
    times[0] = 0.
    return pd.Series(times)

In [14]:
def get_trace(data, id):
    trace = (
        data.query(f"CaseID == '{id}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )
    return trace

In [15]:
def get_timedelta_column(data: pd.DataFrame):
    ids = data["CaseID"].unique()
    timedeltas = []
    for i in range(len(ids)):
        timedeltas.append(
            get_new_timestamp(get_trace(data, ids[i]))
        )
    return pd.concat(timedeltas)
            

In [16]:
timedelta_train_set =  get_timedelta_column(tab_train)

In [17]:
timedelta_train_set

0    0.000000e+00
1    1.908758e+01
2    1.908758e+01
3    1.908758e+01
4    1.908758e+01
         ...     
1    4.222380e-07
2    3.938348e-05
3    1.725527e-03
0    0.000000e+00
1    9.521862e-03
Length: 4493, dtype: float64

In [18]:
mean_timedelta_train = timedelta_train_set.mean()
median_timedelta_train = timedelta_train_set.median()
print(f"Mean Timedelta {mean_timedelta_train}\nMedian Timedelta {median_timedelta_train}")

Mean Timedelta 0.08877813703777664
Median Timedelta 0.01036749493757938


In [19]:
if ACT_TIME_ONLY:
    categorical_columns = ["Activity"]
    real_value_columns = ["time:timestamp"]
    dataset = f"{dataset}_AT_only"
else:
    categorical_columns = dataset_info["categorical"]
    real_value_columns = dataset_info["numerical"]

In [20]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [21]:
def get_freq_dictionary(data: pd.DataFrame, catcol):
    res = {}
    for k in catcol:
        res[k] = data[k].value_counts().to_dict()
    return res

In [22]:
cat_freq = get_freq_dictionary(tab_train, categorical_columns)

In [23]:


def get_tuple_freq(data: pd.DataFrame, k):
    act = data["Activity"].values 
    res = {}
    cat_val = data[k].values 
    for i in range(len(act)):
        if act[i] is not np.nan  and cat_val[i] is not np.nan:
            if act[i] not in res.keys():
                res[act[i]] = {}
            try:
                res[act[i]][cat_val[i]] += 1
            except KeyError:
                res[act[i]][cat_val[i]] = 1
    return res
    

In [24]:


tuple_activity_cat_freq = {}

for c in categorical_columns:
    if c != "Activity":
        tuple_activity_cat_freq[c] = get_tuple_freq(tab_train, c)

In [25]:
def get_mean_value(data: pd.DataFrame, realcol):
    res = {}
    for r in realcol:
        res[r] = data[r].mean()
    return res 

In [26]:
to_check = real_value_columns.copy()
to_check.remove("time:timestamp")
mean_values = get_mean_value(tab_train, to_check)
mean_values

{}

In [27]:
test_types = ["even", "odd", "window", "random", "attr_level"]

In [28]:
def get_masked_test_set(mask_set: pd.DataFrame):
    return mask_set[mask_set["CaseID"].isin(CASE_TEST_IDS)]

In [29]:
masked_test_sets = {
    k : get_masked_test_set(masked_datasets[k]).reset_index(drop=True)
    for k in test_types
}

In [30]:
events_to_repair = {
    k : masked_test_sets[k] != tab_test
    for k in test_types
}


In [31]:
def map_fx(x,category):
    try:
        y = max(tuple_activity_cat_freq[category][x], key=tuple_activity_cat_freq[category][x].get)
    except KeyError:
        y = max(cat_freq[category], key=cat_freq[category].get)
    return y 

In [32]:
def get_accuracy_act_to_cat_attr_level(category):
    predictions = tab_test[events_to_repair["attr_level"][category]]["Activity"].dropna().reset_index(drop=True).apply(
        lambda x: map_fx(x, category)
    )
    labels = tab_test[events_to_repair["attr_level"][category]][category].dropna().reset_index(drop=True)
    val_p = predictions.values
    val_l = labels.values
    c = 0
    for i in range(len(val_l)):
        if val_l[i] == val_p[i]:
            c+=1
    return c / len(labels)

In [47]:
def get_accuracy_simple_freq_attr_level(category):
    most_freq_val = max(cat_freq[category], key=cat_freq[category].get)
    labels = tab_test[events_to_repair["attr_level"][category]][category].dropna().reset_index(drop=True)
    predictions = pd.Series([most_freq_val]*len(labels))
    try: 
        res = (predictions == labels).value_counts()[True] / len(labels)
    except KeyError:
        res = 0.
    return res

In [34]:
def get_MAE_attr_level(column):
    labels = tab_test[events_to_repair["attr_level"][column]][column].dropna().reset_index(drop=True)
    mean_val = mean_values[column]
    predictions = pd.Series([mean_val]*len(labels))
    return (labels-predictions).apply(abs).mean()

In [35]:
def get_accuracy_act_to_cat(category, mask):
    predictions = tab_test[events_to_repair[mask]]["Activity"].dropna().reset_index(drop=True).apply(
        lambda x: map_fx(x, category)
    )
    labels = tab_test[events_to_repair[mask]][category].dropna().reset_index(drop=True)
    val_p = predictions.values
    val_l = labels.values
    c = 0
    for i in range(len(val_l)):
        if val_l[i] == val_p[i]:
            c+=1
    return c / len(labels)

In [48]:
def get_accuracy_simple_freq(category, mask):
    most_freq_val = max(cat_freq[category], key=cat_freq[category].get)
    labels = tab_test[events_to_repair[mask]][category].dropna().reset_index(drop=True)
    predictions = pd.Series([most_freq_val]*len(labels))
    try: 
        res = (predictions == labels).value_counts()[True] / len(labels)
    except KeyError:
        res = 0.
    return res

In [49]:
def get_MAE(column, mask):
    labels = tab_test[events_to_repair[mask]][column].dropna().reset_index(drop=True)
    mean_val = mean_values[column]
    predictions = pd.Series([mean_val]*len(labels))
    return (labels-predictions).apply(abs).mean()

In [38]:
cat_to_check = categorical_columns.copy()
cat_to_check.remove("Activity")

real_to_check = to_check

print(cat_to_check, real_to_check)

['org:resource', 'impact', 'concept:name', 'lifecycle:transition', 'org:group', 'org:role', 'organization country', 'organization involved', 'product', 'resource country'] []


In [50]:
res = {}

for ms in nan_methods:
    res[ms] = {}
    print(f"\n------{ms}-------")
    if ms != "attr_level":
        for c in cat_to_check:
            print(f"Checking {c}...")
            res[ms][c] = {}
            res[ms][c]["Simple_freq"] = get_accuracy_simple_freq(category=c, mask=ms)
            res[ms][c]["Act_to_cat_freq"] = get_accuracy_act_to_cat(category=c, mask=ms)
            print("Done!")
        for r in real_to_check:
            print(f"Checking {r}")
            res[ms][r] = get_MAE(column=r, mask=ms)
            print("Done!")
    else:
        for c in cat_to_check:
            print(f"Checking {c}...")
            res[ms][c] = {}
            res[ms][c]["Simple_freq"] = get_accuracy_simple_freq_attr_level(category=c)
            res[ms][c]["Act_to_cat_freq"] = get_accuracy_act_to_cat_attr_level(category=c)
            print("Done!")
        for r in real_to_check:
            print(f"Checking {r}")
            res[ms][r] = get_MAE_attr_level(column=r)
            print("Done!")
        

with open(f"{dataset}_BASELINE.json", "w") as out:
    json.dump(res, out)


------odd-------
Checking org:resource...
Done!
Checking impact...
Done!
Checking concept:name...
Done!
Checking lifecycle:transition...
Done!
Checking org:group...
Done!
Checking org:role...
Done!
Checking organization country...
Done!
Checking organization involved...
Done!
Checking product...
Done!
Checking resource country...
Done!

------even-------
Checking org:resource...
Done!
Checking impact...
Done!
Checking concept:name...
Done!
Checking lifecycle:transition...
Done!
Checking org:group...
Done!
Checking org:role...
Done!
Checking organization country...
Done!
Checking organization involved...
Done!
Checking product...
Done!
Checking resource country...
Done!

------random-------
Checking org:resource...
Done!
Checking impact...
Done!
Checking concept:name...
Done!
Checking lifecycle:transition...
Done!
Checking org:group...
Done!
Checking org:role...
Done!
Checking organization country...
Done!
Checking organization involved...
Done!
Checking product...
Done!
Checking resou